# 7. Przepływ optyczny


Przepływow optyczny (ang. optical flow) -  technika wizyjna, która pozwala "zobaczyć" i opisać ruch na nagraniu wideo. Mówiąc najprościej: algorytm bierze dwie kolejne klatki filmu i dla każdego piksela (lub wybranej grupy pikseli) stara się wyliczyć wektor przesunięcia – czyli zgadnąć, w którym kierunku i o ile pikseli przesunął się dany fragment obrazu.

w 7.1 imlementujemy metodę blokową:

- wycinamy blok (7x7 z pierwszej klatki video)
- przesuwamy ten kwadracik po malym otoczeniu na drugiej klatce i szukamy meijsca gdzie piksele najbardziej pasuja
- zapisujemy wektor - obliczamy odleglosc miedzy starym a nowym polozeniem bloku - to nasz ruch
- wizualizujemy  - przesuniecie to obraz
- filtrujemy - nakladamy na wyniki poprawke z tzw. maski róznicy obrazow aby pozbyc sie blednych obliczen na gladkich tlach

## 7.1 Metoda Blokowa

In [ ]:
import cv2      
import numpy as np     
import matplotlib.pyplot as plt #

%matplotlib inline

- wczytanie 2 kolejncyh klatek video

In [ ]:
obraz_I = cv2.imread('I.jpg') 
obraz_J = cv2.imread('J.jpg')

skala = 0.5
obraz_I_maly = cv2.resize(obraz_I, (0, 0), fx=skala, fy=skala)
obraz_J_maly = cv2.resize(obraz_J, (0, 0), fx=skala, fy=skala)

I_szary = cv2.cvtColor(obraz_I_maly, cv2.COLOR_BGR2GRAY)
J_szary = cv2.cvtColor(obraz_J_maly, cv2.COLOR_BGR2GRAY)

roznica = cv2.absdiff(I_szary, J_szary)

plt.figure(figsize=(15, 5))
plt.subplot(1, 3, 1), plt.imshow(I_szary, cmap='gray'), plt.title("Klatka I (wcześniejsza)")
plt.subplot(1, 3, 2), plt.imshow(J_szary, cmap='gray'), plt.title("Klatka J (późniejsza)")
plt.subplot(1, 3, 3), plt.imshow(roznica, cmap='gray'), plt.title("Różnica (absdiff)")
plt.show()

algorytm blokowy

In [ ]:
polowa_okna = 5   
zakres_X = 5   
zakres_Y = 5    

wysokosc, szerokosc = I_szary.shape

macierz_u = np.zeros((wysokosc, szerokosc), dtype=np.float32)
macierz_v = np.zeros((wysokosc, szerokosc), dtype=np.float32)

margines_Y = polowa_okna + zakres_Y
margines_X = polowa_okna + zakres_X

# przeszukujemy obraz I (z pominieciem brzegow)
for j in range(margines_Y, wysokosc - margines_Y):
    for i in range(margines_X, szerokosc - margines_X):
        
        # wycinamy blok z pierwszej klatki
        blok_I = np.float32(I_szary[j - polowa_okna : j + polowa_okna + 1, 
                                    i - polowa_okna : i + polowa_okna + 1])
        
        min_dystans = float('inf') # zmienna do szukania najmniejszej roznicy
        najlepsze_dx = 0
        najlepsze_dy = 0
        
        # przeszukujemy otoczenie na drugiej klatce J
        for dy in range(-zakres_Y, zakres_Y + 1):
            for dx in range(-zakres_X, zakres_X + 1):
                
                # wycinamy blok z drugiej klatki (przesuniety o dx, dy)
                blok_J = np.float32(J_szary[j + dy - polowa_okna : j + dy + polowa_okna + 1, 
                                            i + dx - polowa_okna : i + dx + polowa_okna + 1])
                
                # obliczamy "odleglosc" (rozncie) miedzy blokami
                dystans = np.sqrt(np.sum(np.square(blok_J - blok_I)))
                
                # jesli znalezlismy bardziej podobny blok, zapisujemy jego parametry
                if dystans < min_dystans:
                    min_dystans = dystans
                    najlepsze_dx = dx
                    najlepsze_dy = dy
                    
        # zapisujemy najlepsze znalezione przesuniecie dla piksela (j, i)
        macierz_u[j, i] = najlepsze_dx
        macierz_v[j, i] = najlepsze_dy


wizualizacja

In [ ]:
# zamiana wspolrzednych wektora ruchu (u, v) na dlugosc wektora i jego kat (biegunowe)
dlugosc, kat = cv2.cartToPolar(macierz_u, macierz_v)

# wworzymy pusty obraz HSV
obraz_hsv = np.zeros((wysokosc, szerokosc, 3), dtype=np.uint8)

obraz_hsv[..., 0] = np.uint8(kat * 90 / np.pi)

# zamieniamy S z V miejscami aby "brak ruchu" był czarny a nie bialy
obraz_hsv[..., 1] = 255

# V to dlugosc wektora 
obraz_hsv[..., 2] = cv2.normalize(dlugosc, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

obraz_rgb = cv2.cvtColor(obraz_hsv, cv2.COLOR_HSV2RGB)

plt.figure(figsize=(8, 8))
plt.imshow(obraz_rgb)
plt.title("Przepływ optyczny (metoda blokowa)")
plt.axis('off')
plt.show()

Ponieważ metoda blokowa łatwo myli się na jednolitych tłach, wykorzystamy wcześniej policzoną różnicę klatek. Nakładamy filtr, który "przepuści" wektory ruchu tylko tam, gdzie faktycznie zaszła jakakolwiek zmiana pikseli.

In [ ]:
_, maska_binarna = cv2.threshold(roznica, 15, 255, cv2.THRESH_BINARY)

element_strukturalny = np.ones((5, 5), np.uint8)
maska_dylatacja = cv2.dilate(maska_binarna, element_strukturalny, iterations=2)

u_po_filtracji = np.where(maska_dylatacja > 0, macierz_u, 0)
v_po_filtracji = np.where(maska_dylatacja > 0, macierz_v, 0)
dlugosc_f, kat_f = cv2.cartToPolar(u_po_filtracji, v_po_filtracji)

hsv_f = np.zeros((wysokosc, szerokosc, 3), dtype=np.uint8)
hsv_f[..., 0] = np.uint8(kat_f * 90 / np.pi)
hsv_f[..., 1] = 255
hsv_f[..., 2] = cv2.normalize(dlugosc_f, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

rgb_przefiltrowany = cv2.cvtColor(hsv_f, cv2.COLOR_HSV2RGB)

plt.figure(figsize=(8, 8))
plt.imshow(rgb_przefiltrowany)
plt.title("Przepływ optyczny (po nałożeniu maski ruchu)")
plt.axis('off')
plt.show()

Ćwiczenie 7.2 - metoda wieloklasowa - piramida obrazów

Dzięki niej algorytm będzie w stanie wykryć nawet duże i szybkie ruchy bez konieczności drastycznego zwiększania przeszukiwanego okna (co strasznie spowolniłoby obliczenia).

narazie metoda z tego cor obilismy wczesniej w 7.1 czyli metoda blokowa

In [ ]:

# oblicza przeplywa statyczny dlka pary obrazow
def of(J_org, I, J, W2=3, dY=3, dX=3):
    wysokosc, szerokosc = I.shape
    u = np.zeros((wysokosc, szerokosc), dtype=np.float32)
    v = np.zeros((wysokosc, szerokosc), dtype=np.float32)
    
    margines_Y = W2 + dY
    margines_X = W2 + dX
    
    # standardowa metoda blokowa z poprzedniego zadania
    for j in range(margines_Y, wysokosc - margines_Y):
        for i in range(margines_X, szerokosc - margines_X):
            blok_I = np.float32(I[j-W2 : j+W2+1, i-W2 : i+W2+1])
            
            min_dystans = float('inf')
            najlepsze_dx = 0
            najlepsze_dy = 0
            
            for dy in range(-dY, dY+1):
                for dx in range(-dX, dX+1):
                    blok_J = np.float32(J[j+dy-W2 : j+dy+W2+1, i+dx-W2 : i+dx+W2+1])
                    dystans = np.sqrt(np.sum(np.square(blok_J - blok_I)))
                    
                    if dystans < min_dystans:
                        min_dystans = dystans
                        najlepsze_dx = dx
                        najlepsze_dy = dy
                        
            u[j, i] = najlepsze_dx
            v[j, i] = najlepsze_dy

    roznica = cv2.absdiff(I, J_org)
    _, maska = cv2.threshold(roznica, 15, 255, cv2.THRESH_BINARY)
    maska_dylatacja = cv2.dilate(maska, np.ones((5,5), np.uint8), iterations=2)
    
    u = np.where(maska_dylatacja > 0, u, 0)
    v = np.where(maska_dylatacja > 0, v, 0)
    
    return u, v

# zmienia wektory u i v na koło kolorow HsV i wyswietla wynik
def vis_flow(u, v, YX, name):
    wysokosc, szerokosc = YX
    dlugosc, kat = cv2.cartToPolar(u, v)
    
    hsv = np.zeros((wysokosc, szerokosc, 3), dtype=np.uint8)
    hsv[..., 0] = np.uint8(kat * 90 / np.pi)
    hsv[..., 1] = 255
    hsv[..., 2] = cv2.normalize(dlugosc, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    
    rgb = cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)
    
    plt.figure(figsize=(6, 6))
    plt.imshow(rgb)
    plt.title(name)
    plt.axis('off')
    plt.show()

# generuje piramide obrazow pomniejszajac je dwukrotnie w kazdym kroku
def pyramid(im, max_scale):
    images = [im]
    for k in range(1, max_scale):
        # pomniejszamy poprzedni obraz z listy 2-krotnie (fx=0.5, fy=0.5)
        kolejny_obraz = cv2.resize(images[k-1], (0, 0), fx=0.5, fy=0.5)
        images.append(kolejny_obraz)
    return images

w blokowym jesli ruch jest szybko, to obiekt ucieka poza male okienko w ktorym algorytm go szuka 
algorytm wieloklasowy tworzy pare wersji , oryginalne, 2 razy mniejsze itp
najpierw na najmniejszym namierzamy - uruchamiamy tu blokowa
potem do coraz wiekszego obrazu i az do powrotu do oryginalnego 

algorytm wielokskalowy  - algorytm:
- liczy ruch na małym obrazku
- powieksza go
- przesuwa piksele na wiekszym obrazku zgodnie z tym wyliczonym ruchem 
- doszlifowuje wynik

In [ ]:
obraz_I = cv2.imread('I.jpg', cv2.IMREAD_GRAYSCALE)
obraz_J = cv2.imread('J.jpg', cv2.IMREAD_GRAYSCALE)
obraz_I = cv2.resize(obraz_I, (0, 0), fx=0.5, fy=0.5)
obraz_J = cv2.resize(obraz_J, (0, 0), fx=0.5, fy=0.5)

liczba_skal = 3

piramida_I = pyramid(obraz_I, liczba_skal)
piramida_J = pyramid(obraz_J, liczba_skal)

u_suma = None
v_suma = None

for s in range(liczba_skal - 1, -1, -1):
    I_obecne = piramida_I[s]
    J_obecne = piramida_J[s]
    J_org = J_obecne.copy()
    
    h, w = I_obecne.shape
    
    if s != liczba_skal - 1:
        u_suma = cv2.resize(u_suma, (w, h), interpolation=cv2.INTER_LINEAR) * 2.0
        v_suma = cv2.resize(v_suma, (w, h), interpolation=cv2.INTER_LINEAR) * 2.0
        
        #  Warping (kompensacja ruchu) - przesuwamy piksele J zgodnie z u_suma i v_suma
        J_new = np.zeros_like(J_obecne)
        for y in range(h):
            for x in range(w):
                dy = int(round(v_suma[y, x]))
                dx = int(round(u_suma[y, x]))
                nowy_y = y + dy
                nowy_x = x + dx
                
                if 0 <= nowy_y < h and 0 <= nowy_x < w:
                    J_new[y, x] = J_obecne[nowy_y, nowy_x]
                    
        J_obecne = J_new
        
        plt.imshow(J_obecne, cmap='gray')
        plt.title(f"Obraz J po warpingu w skali {s}")
        plt.axis('off')
        plt.show()

    print(f"obliczam przeplyw dla skali: {s} (rozmiar: {w}x{h})...")
    # wyliczamy przeplyw rezydualny (dodatkowy ruch)
    du, dv = of(J_org, I_obecne, J_obecne, W2=3, dY=3, dX=3)
    
    # sumowanie calkowitego przeplywu
    if s == liczba_skal - 1:
        u_suma = du
        v_suma = dv
    else:
        u_suma += du
        v_suma += dv

vis_flow(u_suma, v_suma, obraz_I.shape, "Calkowity przeplyw wieloskalowy (3 skale)")

Ćwiczenie 7.3 - uzycie wbudowanych metod openCV, sprawdziwmy dwa glowne podejscia:
1. gęsty przepływ optyczny - policzony zostanie ruch dla KAŻDEGO piksela na obrazie (użyjemy popularnej metody Farnebacka)

2. Rzadki przepływ optyczny - wygenereujemy na obrazie siatke 10 puntkow np. co 10 pikseli i sprawdizmyr cuh tylko dla nich uzywajac algorytmu Lucasa-Kanade

## Gęsty przepływ optyczny (Metoda Farnebacka)

In [ ]:
obraz_I = cv2.imread('I.jpg', cv2.IMREAD_GRAYSCALE)
obraz_J = cv2.imread('J.jpg', cv2.IMREAD_GRAYSCALE)

skala = 0.5
I_szary = cv2.resize(obraz_I, (0, 0), fx=skala, fy=skala)
J_szary = cv2.resize(obraz_J, (0, 0), fx=skala, fy=skala)
wysokosc, szerokosc = I_szary.shape

przeplyw_farneback = cv2.calcOpticalFlowFarneback(I_szary, J_szary, None, 
                                                  0.5, 3, 15, 3, 5, 1.2, 0)

macierz_u = przeplyw_farneback[..., 0]
macierz_v = przeplyw_farneback[..., 1]

dlugosc, kat = cv2.cartToPolar(macierz_u, macierz_v)

dlugosc[dlugosc > 10] = 10

hsv = np.zeros((wysokosc, szerokosc, 3), dtype=np.uint8)
hsv[..., 0] = np.uint8(kat * 90 / np.pi) 
hsv[..., 1] = 255                       


hsv[..., 2] = cv2.normalize(dlugosc, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

rgb_farneback = cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)

plt.figure(figsize=(8, 8))
plt.imshow(rgb_farneback)
plt.title("Gęsty przepływ optyczny (Metoda Farnebacka)")
plt.axis('off')
plt.show()

## Rzadki przepływ optyczny (Lucas-Kanade na siatce)

In [ ]:
# generujemy rownomeirna siarte punktow (co 10 pikseli)
krok = 10

x, y = np.mgrid[krok//2 : szerokosc : krok, krok//2 : wysokosc : krok]

punkty_poczatkowe = np.vstack((x.flatten(), y.flatten())).T
punkty_poczatkowe = np.float32(punkty_poczatkowe).reshape(-1, 1, 2)


parametry_lk = dict(winSize=(15, 15), 
                    maxLevel=2,      
                    criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 0.03))

nowe_punkty, status, błąd = cv2.calcOpticalFlowPyrLK(I_szary, J_szary, 
                                                     punkty_poczatkowe, None, 
                                                     **parametry_lk)

obraz_wynikowy = cv2.cvtColor(I_szary, cv2.COLOR_GRAY2BGR)

for i, (nowy, stary) in enumerate(zip(nowe_punkty, punkty_poczatkowe)):
    if status[i] == 1:
        a, b = int(nowy[0][0]), int(nowy[0][1])
        c, d = int(stary[0][0]), int(stary[0][1])
        
        dystans = np.sqrt((a - c)**2 + (b - d)**2)
        
        if dystans > 1:
            cv2.line(obraz_wynikowy, (c, d), (a, b), (0, 255, 0), 1)
            cv2.circle(obraz_wynikowy, (a, b), 2, (0, 0, 255), -1)

plt.figure(figsize=(10, 10))
plt.imshow(cv2.cvtColor(obraz_wynikowy, cv2.COLOR_BGR2RGB))
plt.title("Rzadki przepływ optyczny (Lucas-Kanade na siatce)")
plt.axis('off')
plt.show()